In [ ]:
import os

os.environ["GPT_API_KEY"] = "sk-"

from convokitai import Corpus
from convokitai_evaluation import CIG

corpus = Corpus.load("sample_corpus")
corpus.print_summary_stats()

Number of Speakers: 36
Number of Utterances: 220
Number of Conversations: 12
Number of AI Speakers: 36
Number of PrivateAssistants: 0
Number of Supports: 0


In [2]:
cig = CIG(model="gpt-5")
cig.transform(corpus)

10 / 12


In [3]:
# per utterance: 1 (no gain) .. 4 (insightful); None = opening context or too short to rate
convo = next(c for c in corpus.iter_conversations() if c.meta["mediator"] == "constructive")
for utt in convo.get_chronological_utterance_list():
    print(utt.meta["cig"], convo.alias.get(utt.speaker.id, utt.speaker.id), ":", utt.text[:90])

None Mediator (greeting_mediator) : Hello, Goose and Bear, it is a pleasure to have you both here today.
None Bear : I strongly support data center expansion; they are essential for innovation and provide si
None Goose : Economic benefits are great, but not if they risk our power grid and deplete vital local w
3 Bear : Expansion often funds grid upgrades and renewable projects that wouldn't happen otherwise,
1 Mediator (greeting_mediator) : Welcome to the discussion, Goose and Bear.
3 Goose : Funding is helpful, but upgrades take years. We need binding guarantees on resource limits
3 Bear : Strict pre-construction limits risk stalling vital innovation. Flexible partnerships allow
3 Goose : Flexibility often leads to exploitation. We need measurable milestones and accountability 
1 Mediator (greeting_mediator) : It is wonderful to see you both here.
4 Goose : Could we agree on a tiered approval process linked to infrastructure readiness and measura
3 Bear : A tiered approach could work,

In [4]:
# mean per conversation, mediator turns vs participant turns; then per mediator type
summary = cig.summarize(corpus)
summary["mediator"] = summary.conversation_id.map(lambda c: corpus.get_conversation(c).meta["mediator"])
summary.groupby(["mediator", "is_mediator"]).cig_mean.mean()

mediator      is_mediator
constructive  False          2.728001
              True           3.354167
greeting      False          2.812937
              True           1.000000
paraphrase    False          2.745833
              True           1.562500
Name: cig_mean, dtype: float64